# ⚙️ Aplicación a la Ingeniería Mecánica — Mantenimiento predictivo de una fresadora

Este notebook usa **las mismas herramientas del Dashboard ElectroMarket** (pandas, plotly, KPIs, mapas de calor, histogramas)
pero con datos de **una máquina fresadora**: temperaturas, velocidad, torque y desgaste de la herramienta.

**La pregunta:** ¿por qué falla la máquina y cómo podemos verlo venir?

- **Panel de KPIs** de planta: tasa de fallas, potencia, desgaste y **capacidad del proceso (Cp, Cpk)**.
- **12 gráficas** de ingeniería: Pareto de fallas, carta de control, curvas de potencia, histograma con campana de Gauss, correlación y más.
- **Filtro** por calidad del producto (L, M, H).
- **Hallazgos automáticos** y **página web** con todas las gráficas en una pantalla.

**Datos:** *AI4I 2020 Predictive Maintenance Dataset* (UCI Machine Learning Repository, Stephan Matzka, 2020).
Son 10,000 registros de una **fresadora** (máquina que corta metal con una herramienta giratoria), con 5 tipos de falla.
Es una base **sintética**: fue creada para reflejar los datos reales de mantenimiento que se ven en la industria.

## ☁️ ¿Lo abriste en Google Colab?
Corre esta celda primero. En Colab descarga el proyecto de GitHub (con los datos) y activa los filtros.
En tu computadora no hace nada.

In [ ]:
import sys, os, subprocess

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    USUARIO, REPO = "KrystalRou", "Taller_Bob_Dashboard_26"
    if os.path.basename(os.getcwd()) != REPO:
        if not os.path.isdir(REPO):
            subprocess.run(["git", "clone", "-q", f"https://github.com/{USUARIO}/{REPO}.git"], check=True)
        os.chdir(REPO)
    from google.colab import output
    output.enable_custom_widget_manager()   # para que funcionen los filtros
    print("☁️ Listo para Colab ✅")
else:
    print("💻 Corriendo en tu computadora: no hace falta preparar nada ✅")

## 0. Librerías
Las mismas del dashboard de ventas. `numpy` hace la física (potencia, curvas) y la estadística (Gauss, Cp, Cpk).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)
print("Librerías listas ✅")

## 1. Cargar los datos
Pon el archivo `ai4i2020.csv` en la misma carpeta que este notebook (se descarga gratis de UCI).

In [ ]:
candidatos = sorted(Path(".").glob("ai4i*.csv")) + sorted(Path(".").glob("*/ai4i*.csv"))
if not candidatos:
    raise FileNotFoundError("No encontré ai4i2020.csv 🙃  Descárgalo de "
                            "https://archive.ics.uci.edu/dataset/601 y ponlo junto a este notebook.")
ARCHIVO = candidatos[0]
crudo = pd.read_csv(ARCHIVO)
print(f"Archivo encontrado ✅: {ARCHIVO} → {len(crudo):,} registros y {crudo.shape[1]} columnas")
crudo.head()

## 2. Preparar los datos y agregar la física ⚙️
- Pasamos las columnas a nombres en español y las temperaturas de **Kelvin a °C**.
- **ΔT** = temperatura del proceso − temperatura del aire: qué tan bien se disipa el calor.
- **Potencia mecánica**: P = τ · ω, con ω = 2π · rpm / 60 (torque en N·m, velocidad angular en rad/s, potencia en W).
- **Esfuerzo de desgaste** = torque × minutos de desgaste: mide qué tan cansada está la herramienta.

In [ ]:
def buscar(*palabras):
    for c in crudo.columns:
        if all(p.lower() in c.lower() for p in palabras):
            return c
    return None

COL = {
    "calidad":  buscar("type"),
    "t_aire":   buscar("air", "temp"),
    "t_proc":   buscar("process", "temp"),
    "rpm":      buscar("rotational"),
    "torque":   buscar("torque"),
    "desgaste": buscar("tool", "wear"),
    "falla":    buscar("machine", "failure"),
}
faltan = [k for k, v in COL.items() if v is None]
assert not faltan, f"Faltan columnas: {faltan}"

MODOS = {  # columnas de cada tipo de falla en el archivo → nombre en español
    "TWF": "Desgaste de herramienta",
    "HDF": "Disipación de calor",
    "PWF": "Potencia fuera de rango",
    "OSF": "Sobreesfuerzo",
    "RNF": "Aleatoria",
}

df = pd.DataFrame({
    "Registro":      range(1, len(crudo) + 1),
    "Calidad":       crudo[COL["calidad"]].astype(str).str.strip(),
    "T aire (°C)":   crudo[COL["t_aire"]] - 273.15,
    "T proceso (°C)": crudo[COL["t_proc"]] - 273.15,
    "Velocidad (rpm)": crudo[COL["rpm"]],
    "Torque (N·m)":  crudo[COL["torque"]],
    "Desgaste (min)": crudo[COL["desgaste"]],
    "Falla":         crudo[COL["falla"]].astype(int),
})
for clave, nombre in MODOS.items():
    df[nombre] = crudo[clave].astype(int) if clave in crudo else 0

df["ΔT (°C)"] = df["T proceso (°C)"] - df["T aire (°C)"]
df["Potencia (W)"] = df["Torque (N·m)"] * df["Velocidad (rpm)"] * 2 * np.pi / 60
df["Esfuerzo desgaste (N·m·min)"] = df["Torque (N·m)"] * df["Desgaste (min)"]
df["Estado"] = np.where(df["Falla"] == 1, "Falla", "Sin falla")
df["Calidad"] = df["Calidad"].map({"L": "L (baja)", "M": "M (media)", "H": "H (alta)"}).fillna(df["Calidad"])

print("Datos listos ✅")
print(f"Fallas: {df['Falla'].sum():,} de {len(df):,} registros ({df['Falla'].mean():.1%})")
df.head()

## 3. Panel de KPIs de planta 🎯
Igual que en ventas, pero con indicadores de ingeniería:

- **Tasa de fallas** contra una meta (velocímetro).
- **Cp y Cpk** (capacidad del proceso): ¿la temperatura del proceso cabe dentro de sus límites de especificación?
  - **Cp** compara el ancho de la tolerancia con la variación del proceso: Cp = (LSE − LIE) / 6σ.
  - **Cpk** además castiga que el proceso esté descentrado: Cpk = mín(LSE − μ, μ − LIE) / 3σ.
  - Regla de la industria: **Cpk ≥ 1.33** es un proceso capaz.

Los límites LIE y LSE son **de ejemplo**: cámbialos por los de tu pieza o proceso.

In [ ]:
META_FALLAS = 2.0        # % máximo de registros con falla que aceptamos
LIE, LSE = 31.0, 43.0    # límites de especificación de la temperatura del proceso (°C), de ejemplo
META_CPK = 1.33

def capacidad(x, lie=LIE, lse=LSE):
    mu, s = x.mean(), x.std()
    return (lse - lie) / (6 * s), min(lse - mu, mu - lie) / (3 * s)

def panel_kpis(d, titulo="KPIs de la máquina"):
    cp, cpk = capacidad(d["T proceso (°C)"])
    fig = make_subplots(rows=2, cols=4, specs=[[{"type": "indicator"}] * 4] * 2,
                        vertical_spacing=0.25, horizontal_spacing=0.06)

    def numero(valor, nombre, fila, col, sufijo="", formato=",.0f"):
        fig.add_trace(go.Indicator(mode="number", value=valor,
                                   number=dict(suffix=sufijo, valueformat=formato, font_size=34),
                                   title=dict(text=nombre, font_size=15)), fila, col)

    def velocimetro(valor, nombre, fila, col, meta, maximo, sufijo="", menos_es_mejor=False, formato=".1f"):
        verde, rojo = "#d4edda", "#f8d7da"
        pasos = ([dict(range=[0, meta], color=verde), dict(range=[meta, maximo], color=rojo)] if menos_es_mejor
                 else [dict(range=[0, meta], color=rojo), dict(range=[meta, maximo], color=verde)])
        fig.add_trace(go.Indicator(
            mode="gauge+number+delta", value=valor,
            number=dict(suffix=sufijo, valueformat=formato, font_size=30),
            delta=dict(reference=meta, valueformat=formato,
                       increasing_color="#d62728" if menos_es_mejor else "#2ca02c",
                       decreasing_color="#2ca02c" if menos_es_mejor else "#d62728"),
            title=dict(text=nombre, font_size=15),
            gauge=dict(axis=dict(range=[0, maximo]), bar=dict(color="#1f3b73"), steps=pasos,
                       threshold=dict(line=dict(color="black", width=4), value=meta))), fila, col)

    numero(len(d), "Registros", 1, 1)
    numero(d["Falla"].sum(), "Fallas", 1, 2)
    numero(d["Potencia (W)"].mean() / 1000, "Potencia media", 1, 3, " kW", ".2f")
    numero(d["Desgaste (min)"].mean(), "Desgaste medio", 1, 4, " min", ".0f")
    velocimetro(d["Falla"].mean() * 100, f"Tasa de fallas · meta {META_FALLAS}%", 2, 1,
                META_FALLAS, 10, "%", menos_es_mejor=True)
    velocimetro(cpk, f"Cpk temperatura · meta {META_CPK}", 2, 2, META_CPK, 3, formato=".2f")
    numero(cp, "Cp temperatura", 2, 3, formato=".2f")
    numero(d["ΔT (°C)"].mean(), "ΔT medio", 2, 4, " °C", ".1f")
    return fig.update_layout(title=titulo, height=480, margin=dict(t=90, b=20))

panel_kpis(df).show()

## 4. Doce gráficas de ingeniería 📈
Cada gráfica responde una pregunta de mantenimiento o de calidad. Igual que en ventas, cada una es una **función**
que recibe los datos y regresa la figura, así el filtro y la página web las pueden volver a dibujar.

In [ ]:
COLORES = px.colors.qualitative.Set2
COLOR_ESTADO = {"Sin falla": "#8da0cb", "Falla": "#d62728"}
ORDEN_CALIDAD = ["L (baja)", "M (media)", "H (alta)"]

def gauss(x, mu, s):
    return np.exp(-((x - mu) / s) ** 2 / 2) / (s * np.sqrt(2 * np.pi))

# 1) Pareto: qué tipos de falla pesan más (la herramienta clásica de calidad: 80/20)
def g_pareto(d):
    t = d[list(MODOS.values())].sum().sort_values(ascending=False)
    t = t[t > 0]
    if t.empty:
        return None
    acum = t.cumsum() / t.sum() * 100
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=t.index, y=t.values, name="Fallas", marker_color=COLORES[1], text=t.values)
    fig.add_scatter(x=t.index, y=acum.values, name="% acumulado", mode="lines+markers",
                    line=dict(color="#1f3b73"), secondary_y=True)
    fig.add_hline(y=80, line_dash="dot", line_color="gray", secondary_y=True)
    fig.update_yaxes(title_text="Número de fallas", secondary_y=False)
    fig.update_yaxes(title_text="% acumulado", range=[0, 105], secondary_y=True)
    return fig.update_layout(title="1. ¿Qué falla más? · Diagrama de Pareto", showlegend=False)

# 2) Barras: tasa de falla según la calidad del producto
def g_calidad(d):
    t = d.groupby("Calidad", as_index=False)["Falla"].mean()
    t["Tasa de falla (%)"] = t["Falla"] * 100
    fig = px.bar(t, x="Calidad", y="Tasa de falla (%)", text_auto=".1f", color="Calidad",
                 category_orders={"Calidad": ORDEN_CALIDAD}, color_discrete_sequence=COLORES,
                 title="2. Tasa de falla por calidad del producto · Barras")
    fig.add_hline(y=META_FALLAS, line_dash="dash", line_color="red", annotation_text=f"meta {META_FALLAS}%")
    return fig.update_layout(showlegend=False, xaxis_title="")

# 3) Histograma + Gauss + límites de especificación: control de calidad (Cp, Cpk)
def g_capacidad(d):
    x = d["T proceso (°C)"]
    mu, s = x.mean(), x.std()
    cp, cpk = capacidad(x)
    fig = px.histogram(x=x, nbins=50, histnorm="probability density", color_discrete_sequence=[COLORES[2]],
                       title="3. Capacidad del proceso: temperatura · Histograma con campana de Gauss")
    xs = np.linspace(min(x.min(), LIE - 1), max(x.max(), LSE + 1), 300)
    fig.add_scatter(x=xs, y=gauss(xs, mu, s), mode="lines", line=dict(color="red", width=3), name="Gauss")
    for v, n in [(LIE, "LIE"), (LSE, "LSE")]:
        fig.add_vline(x=v, line_dash="dash", line_color="black", annotation_text=n)
    fig.add_annotation(text=f"μ = {mu:.2f} °C · σ = {s:.2f} · Cp = {cp:.2f} · Cpk = {cpk:.2f}",
                       xref="paper", yref="paper", x=0.02, y=0.98, showarrow=False)
    return fig.update_layout(showlegend=False, xaxis_title="Temperatura del proceso (°C)", yaxis_title="Densidad")

# 4) Histograma + Gauss del torque, separado por estado: ¿las fallas viven en las colas?
def g_torque(d):
    fig = px.histogram(d, x="Torque (N·m)", color="Estado", nbins=60, histnorm="probability density",
                       barmode="overlay", opacity=0.6, color_discrete_map=COLOR_ESTADO,
                       title="4. Torque: la campana normal y las fallas en las colas · Histograma")
    x = d["Torque (N·m)"]
    xs = np.linspace(x.min(), x.max(), 300)
    fig.add_scatter(x=xs, y=gauss(xs, x.mean(), x.std()), mode="lines", name="Gauss",
                    line=dict(color="black", width=2, dash="dot"))
    return fig.update_layout(yaxis_title="Densidad")

# 5) Dispersión con física: torque vs velocidad y curvas de potencia constante (P = τ·ω)
def g_potencia(d):
    m = d.sample(min(len(d), 4000), random_state=1).sort_values("Estado", ascending=False)
    fig = px.scatter(m, x="Velocidad (rpm)", y="Torque (N·m)", color="Estado", opacity=0.6,
                     color_discrete_map=COLOR_ESTADO,
                     title="5. Torque vs. velocidad con curvas de potencia constante · Dispersión")
    rpm = np.linspace(d["Velocidad (rpm)"].min(), d["Velocidad (rpm)"].max(), 200)
    for P, nombre in [(3500, "3.5 kW"), (9000, "9 kW")]:
        tau = P / (rpm * 2 * np.pi / 60)
        ok = tau <= d["Torque (N·m)"].max() * 1.05
        fig.add_scatter(x=rpm[ok], y=tau[ok], mode="lines", name=f"P = {nombre}",
                        line=dict(color="black", dash="dash"))
    return fig

# 6) Cajas: desgaste de la herramienta en máquinas con y sin falla
def g_desgaste(d):
    fig = px.box(d, x="Estado", y="Desgaste (min)", color="Estado", color_discrete_map=COLOR_ESTADO,
                 category_orders={"Estado": ["Sin falla", "Falla"]}, points=False,
                 title="6. ¿Las herramientas gastadas fallan más? · Cajas")
    return fig.update_layout(showlegend=False, xaxis_title="")

# 7) Línea: tasa de falla según los minutos de desgaste (zona de desgaste de la curva de la bañera)
def g_banera(d):
    bins = np.arange(0, d["Desgaste (min)"].max() + 21, 20)
    t = d.groupby(pd.cut(d["Desgaste (min)"], bins, include_lowest=True), observed=True)["Falla"].agg(["mean", "size"])
    t = t[t["size"] >= 20]
    x = [iv.mid for iv in t.index]
    fig = px.line(x=x, y=t["mean"] * 100, markers=True,
                  title="7. La falla crece con el desgaste · Línea (zona final de la curva de la bañera)")
    fig.update_traces(line_color="#d62728")
    return fig.update_layout(xaxis_title="Desgaste de la herramienta (min)", yaxis_title="Tasa de falla (%)")

# 8) Mapa de calor: correlación entre variables físicas y la falla
def g_correlacion(d):
    cols = ["T aire (°C)", "T proceso (°C)", "ΔT (°C)", "Velocidad (rpm)", "Torque (N·m)",
            "Potencia (W)", "Desgaste (min)", "Falla"]
    t = d[cols].corr()
    fig = px.imshow(t.round(2), text_auto=".2f", color_continuous_scale="RdBu", zmin=-1, zmax=1,
                    aspect="auto", title="8. ¿Qué variables se mueven juntas? · Mapa de calor de correlación")
    return fig.update_layout(coloraxis_colorbar_title="r")

# 9) Mapa de calor: tasa de falla por torque y desgaste (sobreesfuerzo)
def g_mapa_falla(d):
    t = d.assign(
        T=pd.cut(d["Torque (N·m)"], [0, 20, 30, 40, 50, 60, 80]),
        D=pd.cut(d["Desgaste (min)"], [-1, 50, 100, 150, 200, 260]),
    ).pivot_table(index="T", columns="D", values="Falla", aggfunc="mean", observed=False) * 100
    t.index = [f"{int(i.left)}–{int(i.right)} N·m" for i in t.index]
    t.columns = [f"{max(int(c.left), 0)}–{int(c.right)} min" for c in t.columns]
    fig = px.imshow(t.round(1), text_auto=".0f", aspect="auto", color_continuous_scale="Reds",
                    title="9. % de falla por torque y desgaste · Mapa de calor")
    return fig.update_layout(coloraxis_colorbar_title="% falla", xaxis_title="Desgaste", yaxis_title="Torque")

# 10) Carta de control X̄ (control estadístico de procesos): ¿la temperatura se sale de control?
def g_carta_control(d, n=50):
    g = d.reset_index(drop=True).groupby(np.arange(len(d)) // n)["T proceso (°C)"]
    t = pd.DataFrame({"media": g.mean(), "size": g.size()})
    t = t[t["size"] == n]
    if len(t) < 5:
        return None
    centro = t["media"].mean()
    sigma = g.std().mean() / np.sqrt(n)   # variación DENTRO de cada subgrupo (como se hace en control estadístico)
    lcs, lci = centro + 3 * sigma, centro - 3 * sigma
    fuera = (t["media"] > lcs) | (t["media"] < lci)
    fig = go.Figure()
    fig.add_scatter(x=t.index + 1, y=t["media"], mode="lines+markers", name="Media del subgrupo",
                    marker=dict(color=np.where(fuera, "#d62728", "#1f3b73")), line=dict(color="#8da0cb"))
    for v, n_, c in [(lcs, "LCS", "red"), (centro, "Centro", "green"), (lci, "LCI", "red")]:
        fig.add_hline(y=v, line_dash="dash" if n_ != "Centro" else "solid", line_color=c,
                      annotation_text=f"{n_} {v:.2f}")
    return fig.update_layout(title=f"10. Carta de control X̄ de temperatura (subgrupos de {n}) · Control estadístico",
                             xaxis_title="Subgrupo", yaxis_title="Media T proceso (°C)", showlegend=False)

# 11) Dispersión: ΔT vs velocidad — la zona donde el calor no se disipa
def g_calor(d):
    m = d.sample(min(len(d), 4000), random_state=2).sort_values("Estado", ascending=False)
    fig = px.scatter(m, x="Velocidad (rpm)", y="ΔT (°C)", color="Estado", opacity=0.6,
                     color_discrete_map=COLOR_ESTADO,
                     title="11. Disipación de calor: ΔT vs. velocidad · Dispersión")
    fig.add_shape(type="rect", x0=d["Velocidad (rpm)"].min(), x1=1380, y0=d["ΔT (°C)"].min(), y1=8.6,
                  line=dict(color="red", dash="dash"), fillcolor="rgba(214,39,40,0.08)")
    fig.add_annotation(x=1380, y=8.6, text="zona de riesgo: ΔT < 8.6 °C y < 1380 rpm",
                       showarrow=False, xanchor="left", yanchor="bottom", font=dict(color="red"))
    return fig

# 12) Áreas apiladas: cómo cambia la mezcla de fallas a lo largo de la producción
def g_mezcla(d, bloque=500):
    cols = [c for c in MODOS.values() if d[c].sum() > 0]
    if not cols:
        return None
    t = d.groupby((d["Registro"] - 1) // bloque)[cols].sum()
    t.index = (t.index + 1) * bloque
    t = t.reset_index(names="Registros").melt(id_vars="Registros", var_name="Tipo de falla", value_name="Fallas")
    fig = px.area(t, x="Registros", y="Fallas", color="Tipo de falla", line_shape="spline",
                  color_discrete_sequence=COLORES,
                  title=f"12. Fallas a lo largo de la producción (cada {bloque} registros) · Áreas apiladas")
    return fig.update_layout(xaxis_title="Registro de producción")

GRAFICAS = [g_pareto, g_calidad, g_capacidad, g_torque, g_potencia, g_desgaste,
            g_banera, g_correlacion, g_mapa_falla, g_carta_control, g_calor, g_mezcla]

def mostrar(g, d, alto=440):
    fig = g(d)
    if fig is not None:
        fig.update_layout(height=alto, template="plotly_white").show()

for g in GRAFICAS:
    mostrar(g, df)

## 5. Dashboard interactivo con filtro 🎛️
Elige la **calidad del producto**: los KPIs y las 12 gráficas se recalculan solos.

In [ ]:
f_calidad = widgets.Dropdown(options=["Todas"] + [c for c in ORDEN_CALIDAD if c in df["Calidad"].unique()],
                             description="Calidad")
salida = widgets.Output()

def filtrar():
    return df if f_calidad.value == "Todas" else df[df["Calidad"] == f_calidad.value]

def dibujar(_=None):
    d = filtrar()
    with salida:
        salida.clear_output(wait=True)
        panel_kpis(d, f"KPIs de la máquina · calidad: {f_calidad.value}").show()
        for g in GRAFICAS:
            mostrar(g, d, alto=420)

f_calidad.observe(dibujar, names="value")
display(HTML("<h2 style='font-family:sans-serif;color:#1f3b73'>⚙️ Mantenimiento predictivo de una fresadora</h2>"),
        f_calidad, salida)
dibujar()

## 6. Hallazgos automáticos
Python escribe las conclusiones de ingeniería buscando máximos, mínimos y umbrales.

In [ ]:
modos = df[list(MODOS.values())].sum().sort_values(ascending=False)
cp, cpk = capacidad(df["T proceso (°C)"])
por_calidad = df.groupby("Calidad")["Falla"].mean().sort_values(ascending=False) * 100
gastadas = df[df["Desgaste (min)"] >= 200]["Falla"].mean() * 100
nuevas = df[df["Desgaste (min)"] < 200]["Falla"].mean() * 100
r = df[["Torque (N·m)", "Velocidad (rpm)"]].corr().iloc[0, 1]
top2 = modos.head(2)

print(f"🔧 Tasa de fallas: {df['Falla'].mean():.1%}  (meta: {META_FALLAS}%)")
print(f"📊 Pareto: '{top2.index[0]}' y '{top2.index[1]}' suman {top2.sum() / modos.sum():.0%} de las fallas")
print(f"🏭 Calidad con más fallas: {por_calidad.index[0]} ({por_calidad.iloc[0]:.1f}%)")
print(f"⏳ Herramientas con 200 min o más de desgaste fallan {gastadas:.1f}% vs {nuevas:.1f}% del resto")
print(f"🌡️ Temperatura del proceso: Cp = {cp:.2f}, Cpk = {cpk:.2f} → "
      + ("proceso capaz ✅" if cpk >= META_CPK else "proceso NO capaz con estos límites ⚠️"))
print(f"⚙️ Torque y velocidad tienen correlación r = {r:.2f}: a más velocidad, menos torque (P = τ·ω)")

## 7. Guardar el dashboard como página web
Crea `aplicacion_ingenieria_mecanica.html` con los KPIs y las 12 gráficas **en una sola pantalla**.
Ábrelo con doble clic: **toca una gráfica** para verla en grande, ◀ ▶ para pasar a la siguiente y **Esc** para regresar.

In [ ]:
import sys
import plotly.io as pio

PLANTILLA = r"""<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Aplicación a la Ingeniería Mecánica</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  * { box-sizing: border-box; }
  body { margin: 0; font-family: -apple-system, "Segoe UI", Roboto, sans-serif; background: #f4f6fa; color: #1c2533; }
  header { height: 56px; display: flex; align-items: center; justify-content: space-between; padding: 0 20px; }
  header h1 { font-size: 20px; margin: 0; color: #1f3b73; }
  header span { font-size: 13px; color: #5b6575; }
  .grid { display: grid; gap: 10px; padding: 0 14px 14px;
          grid-template-columns: repeat(4, 1fr);
          grid-auto-rows: max(170px, calc((100vh - 56px - 14px - 30px) / 4)); }
  .card { background: #fff; border: 1px solid #dfe3ea; border-radius: 10px; padding: 8px 10px 4px;
          display: flex; flex-direction: column; cursor: zoom-in; transition: box-shadow .15s, transform .15s; min-width: 0; }
  .card:hover { box-shadow: 0 6px 18px rgba(31,59,115,.18); transform: translateY(-2px); }
  .card h3 { margin: 0 0 4px; font-size: 13px; font-weight: 600; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
  .card h3::after { content: " ⤢"; color: #8a94a6; }
  .card .plot { flex: 1; min-height: 0; pointer-events: none; }
  .card.kpis { grid-column: span 2; grid-row: span 2; }
  #vista { position: fixed; inset: 0; background: #fff; display: none; flex-direction: column; z-index: 10; }
  #vista.abierta { display: flex; }
  #vista .barra { height: 52px; display: flex; align-items: center; gap: 10px; padding: 0 16px; border-bottom: 1px solid #dfe3ea; }
  #vista .barra h2 { flex: 1; font-size: 17px; margin: 0; }
  #vista button { font-size: 14px; padding: 6px 12px; border: 1px solid #c8cfda; border-radius: 8px; background: #fff; cursor: pointer; }
  #vista button:hover { background: #eef2f8; }
  #grande { flex: 1; min-height: 0; }
  @media (max-width: 900px) {
    .grid { grid-template-columns: repeat(2, 1fr); grid-auto-rows: 220px; }
  }
</style></head>
<body>
<header><h1>⚙️ Aplicación a la Ingeniería Mecánica · Fresadora (datos AI4I 2020)</h1><span>__FILTROS__ · toca una gráfica para verla en grande</span></header>
<div class="grid" id="grid"></div>
<div id="vista">
  <div class="barra"><button id="ant">◀</button><h2 id="titulo"></h2><button id="sig">▶</button><button id="cerrar">✕ Cerrar (Esc)</button></div>
  <div id="grande"></div>
</div>
<script>
const FIGURAS = [__KPIS__].concat(__FIGS__);
const titulo = f => (f.layout.title && (f.layout.title.text || f.layout.title)) || "";
const grid = document.getElementById("grid");
const filas = Math.ceil((FIGURAS.length - 1 + 4) / 4);   // 4 columnas; los KPIs ocupan 4 lugares
if (window.innerWidth > 900) grid.style.gridAutoRows = `max(150px, calc((100vh - 100px) / ${filas}))`;
const vista = document.getElementById("vista");
const grande = document.getElementById("grande");
let actual = -1;

FIGURAS.forEach((fig, i) => {
  const card = document.createElement("div");
  card.className = "card" + (i === 0 ? " kpis" : "");
  card.innerHTML = `<h3>${titulo(fig)}</h3><div class="plot"></div>`;
  card.onclick = () => abrir(i);
  grid.appendChild(card);
  const div = card.querySelector(".plot");
  const layout = Object.assign({}, fig.layout, {
    title: null, autosize: true, height: div.clientHeight || 160,
    margin: i === 0 ? {l: 10, r: 10, t: 10, b: 10} : {l: 34, r: 8, t: 6, b: 26},
    showlegend: false, font: {size: 9}
  });
  if (i > 0) layout.annotations = [];
  let data = fig.data;
  if (i === 0) {  // KPIs: números más chicos para que quepan en la miniatura
    data = fig.data.map(t => Object.assign({}, t, {
      number: Object.assign({}, t.number, {font: {size: t.gauge ? 20 : 26}}),
      delta: Object.assign({}, t.delta, {font: {size: 11}}),
      title: Object.assign({}, t.title, {font: {size: 12}})
    }));
  }
  Plotly.newPlot(div, data, layout, {staticPlot: true, responsive: true, displayModeBar: false});
});

function abrir(i) {
  actual = (i + FIGURAS.length) % FIGURAS.length;
  const fig = FIGURAS[actual];
  document.getElementById("titulo").textContent = titulo(fig);
  vista.classList.add("abierta");
  Plotly.purge(grande);
  const layout = Object.assign({}, fig.layout, {title: null, autosize: true, height: null, width: null});
  Plotly.newPlot(grande, fig.data, layout, {responsive: true, displaylogo: false});
}
function cerrar() { vista.classList.remove("abierta"); Plotly.purge(grande); actual = -1; }
document.getElementById("cerrar").onclick = cerrar;
document.getElementById("ant").onclick = () => abrir(actual - 1);
document.getElementById("sig").onclick = () => abrir(actual + 1);
document.addEventListener("keydown", e => {
  if (actual < 0) return;
  if (e.key === "Escape") cerrar();
  if (e.key === "ArrowRight") abrir(actual + 1);
  if (e.key === "ArrowLeft") abrir(actual - 1);
});
</script>
</body></html>"""


def exportar_html(nombre="aplicacion_ingenieria_mecanica.html"):
    """Página web: todas las gráficas en una sola pantalla; al tocar una se abre en grande."""
    d = filtrar()   # respeta el filtro de calidad
    kpis = panel_kpis(d, f"KPIs de la máquina · calidad: {f_calidad.value}").update_layout(template="plotly_white")
    figuras = [f.update_layout(template="plotly_white") for f in (g(d) for g in GRAFICAS) if f is not None]
    pagina = (PLANTILLA
              .replace("__FILTROS__", f"Calidad: {f_calidad.value}")
              .replace("__KPIS__", pio.to_json(kpis))
              .replace("__FIGS__", "[" + ",".join(pio.to_json(f) for f in figuras) + "]"))
    with open(nombre, "w", encoding="utf-8") as f:
        f.write(pagina)
    if "google.colab" in sys.modules:   # en Colab, la página se descarga a tu computadora
        from google.colab import files
        files.download(nombre)
    return nombre

print("Guardado ✅:", exportar_html(), "→ ábrelo con doble clic en tu navegador")


## 🚀 Retos para seguir practicando
1. Cambia **LIE y LSE** por los límites reales de una pieza que conozcas y observa cómo cambia el **Cpk**.
2. Agrega un filtro de **tipo de falla** (pista: copia cómo se hizo `f_calidad`).
3. Haz una carta de control **R** (rangos) junto a la X̄.
4. Entrena un modelo que prediga la falla con `scikit-learn` (`RandomForestClassifier`) usando torque, velocidad, ΔT y desgaste.
5. Pídele a **IBM Bob** que agregue una gráfica nueva: por ejemplo, la eficiencia de potencia por calidad de producto.